# Assignment 10: Comprehensive PyTorch Challenge (100 points)

**Unit 06: Programming PyTorch | AI 310**

This is a contest-style multi-part problem that tests all PyTorch skills from Unit 06. It simulates the structure of a USAAIO Round 2 problem: progressive parts that build on each other, mixing coding and conceptual tasks.

**Scenario**: You are building a **neural ODE solver** that learns to approximate the solution to an ordinary differential equation. This combines tensors, autograd, custom modules, loss functions, and training loops.

**Notation**:
- $u(t)$ = solution to the ODE at time $t$
- $f(u, t)$ = the right-hand side of the ODE: $\frac{du}{dt} = f(u, t)$
- $\theta$ = neural network parameters

In [ ]:
"""DO NOT MAKE ANY CHANGE IN THIS CELL."""
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import Dataset, DataLoader

torch.manual_seed(42)

**WARNING**: Do not import any additional libraries beyond those provided above.

---

## Part 1 (10 points, non-coding)

Consider the ODE: $\frac{du}{dt} = -\lambda u$, with initial condition $u(0) = u_0$.

1. What is the exact analytical solution $u(t)$?
2. For $\lambda = 1, u_0 = 1$, what is $u(2)$?
3. If we discretize this with Euler's method (step size $h$): $u_{n+1} = u_n + h \cdot f(u_n, t_n)$, what is the stability condition on $h$?

### WRITE YOUR SOLUTION HERE ###

1. *Your answer here*

2. *Your answer here*

3. *Your answer here*

""" END OF THIS PART """

---

## Part 2 (15 points, coding)

Implement **Euler's method** as a differentiable PyTorch function.

Given an ODE $\frac{du}{dt} = f(u, t)$ with initial condition $u(t_0) = u_0$, integrate from $t_0$ to $t_f$ using $N$ steps.

$$u_{n+1} = u_n + h \cdot f(u_n, t_n), \quad h = \frac{t_f - t_0}{N}$$

The function must support autograd (gradients must flow through the integration).

In [ ]:
### WRITE YOUR SOLUTION HERE ###

def euler_integrate(f, u0, t0, tf, N):
    """
    Euler's method for ODE integration.
    
    Args:
        f: callable(u, t) -> du/dt, where u and t are tensors
        u0: initial condition tensor, shape (D,) or (B, D)
        t0: initial time (float)
        tf: final time (float)
        N: number of integration steps (int)
    
    Returns:
        u_final: solution at time tf, same shape as u0
        trajectory: list of (t, u) pairs for all steps
    """
    pass

In [ ]:
""" END OF THIS PART """
# Test: du/dt = -u, u(0) = 1, exact solution u(t) = exp(-t)
f_decay = lambda u, t: -u
u0 = torch.tensor([1.0], requires_grad=True)
u_final, traj = euler_integrate(f_decay, u0, 0.0, 2.0, N=1000)

exact = np.exp(-2.0)
assert abs(u_final.item() - exact) < 0.01, \
    f"Expected ~{exact:.4f}, got {u_final.item():.4f}"

# Verify gradient flows through
u_final.backward()
assert u0.grad is not None, "Gradients must flow through Euler integration"
print(f"Part 2 passed! u(2) = {u_final.item():.4f} (exact: {exact:.4f})")

---

## Part 3 (20 points, coding)

Build a **Neural ODE** module. Instead of knowing $f(u, t)$ analytically, we learn it with a neural network:

$$\frac{du}{dt} = f_\theta(u, t)$$

where $f_\theta$ is a neural network with parameters $\theta$.

Implement `NeuralODE` as an `nn.Module` that:
- Contains a neural network `f_net` that takes `[u, t]` as input
- Uses your `euler_integrate` from Part 2
- `forward(u0, t0, tf, N)` returns the solution at time `tf`

In [ ]:
### WRITE YOUR SOLUTION HERE ###

class NeuralODE(nn.Module):
    def __init__(self, state_dim, hidden_dim=64):
        """
        Args:
            state_dim: dimension of u (the ODE state)
            hidden_dim: hidden layer size for the neural network f
        """
        pass
    
    def forward(self, u0, t0, tf, N=100):
        """
        Integrate from t0 to tf starting from u0.
        
        Args:
            u0: (B, state_dim) initial conditions
            t0: float, initial time
            tf: float, final time
            N: number of Euler steps
        
        Returns:
            u_final: (B, state_dim) solution at tf
        """
        pass

In [ ]:
""" END OF THIS PART """
torch.manual_seed(42)
node = NeuralODE(state_dim=2, hidden_dim=32)

u0 = torch.randn(8, 2)  # batch of 8, state dim 2
u_final = node(u0, 0.0, 1.0, N=50)

assert u_final.shape == (8, 2), f"Expected (8, 2), got {u_final.shape}"

# Verify gradient flow
loss = u_final.sum()
loss.backward()
has_grads = all(p.grad is not None for p in node.parameters())
assert has_grads, "Gradients must flow to network parameters"

param_count = sum(p.numel() for p in node.parameters())
print(f"Part 3 passed! NeuralODE has {param_count} parameters")

---

## Part 4 (15 points, coding)

Create a **training dataset** for the Neural ODE.

The target ODE is the 2D spiral: $\frac{d}{dt}\begin{pmatrix}x\\y\end{pmatrix} = \begin{pmatrix}-0.1x - y \\ x - 0.1y\end{pmatrix}$

This creates a spiral trajectory. Generate training data:
1. Sample 200 initial conditions: $u_0 \sim \mathcal{N}(0, 1)$, shape `(200, 2)`
2. For each $u_0$, compute the true solution at $t = 1.0$ using `euler_integrate` with $N = 1000$ (high accuracy)
3. Create a Dataset returning `(u0, u_target)` pairs

In [ ]:
"""DO NOT MAKE ANY CHANGE IN THIS CELL."""
def spiral_dynamics(u, t):
    """True ODE: du/dt = [-0.1*x - y, x - 0.1*y]"""
    x, y = u[..., 0:1], u[..., 1:2]
    return torch.cat([-0.1 * x - y, x - 0.1 * y], dim=-1)

In [ ]:
### WRITE YOUR SOLUTION HERE ###

class SpiralDataset(Dataset):
    def __init__(self, n_samples=200, t_final=1.0, n_euler_steps=1000):
        pass
    
    def __len__(self):
        pass
    
    def __getitem__(self, idx):
        # Return (u0, u_target) both of shape (2,)
        pass

In [ ]:
""" END OF THIS PART """
torch.manual_seed(42)
dataset = SpiralDataset(200, t_final=1.0)
assert len(dataset) == 200

u0, u_target = dataset[0]
assert u0.shape == (2,), f"u0 shape: {u0.shape}"
assert u_target.shape == (2,), f"u_target shape: {u_target.shape}"
assert not torch.allclose(u0, u_target), "u0 and u_target should be different"

print(f"Part 4 passed! Dataset: {len(dataset)} samples")
print(f"Example: u0={u0.tolist()}, u_target={u_target.tolist()}")

---

## Part 5 (20 points, coding)

**Train the Neural ODE** to learn the spiral dynamics.

Training objective: minimize $\mathcal{L} = \frac{1}{B}\sum_i \|\hat{u}(t_f; u_0^{(i)}) - u^{\text{target}(i)}\|^2$

where $\hat{u}(t_f; u_0)$ is the Neural ODE's prediction starting from $u_0$.

Requirements:
- Use your NeuralODE from Part 3 (state_dim=2, hidden_dim=64)
- Optimizer: Adam with lr=1e-3
- Train for 200 epochs with batch_size=32
- Use N=50 Euler steps during training (faster than N=1000)
- After training, evaluate on a held-out test set using N=200 steps
- Store final test MSE as `test_mse`

In [ ]:
### WRITE YOUR SOLUTION HERE ###

torch.manual_seed(42)

# 1. Create train and test datasets
# 2. Build NeuralODE model
# 3. Training loop (200 epochs)
# 4. Evaluate on test set

test_mse = ...  # float

In [ ]:
""" END OF THIS PART """
assert isinstance(test_mse, float)
assert test_mse < 0.5, f"Test MSE too high: {test_mse:.4f}. Model should learn the spiral dynamics."
print(f"Part 5 passed! Test MSE: {test_mse:.6f}")

---

## Part 6 (20 points, coding)

**Multi-step prediction and analysis.**

Using your trained NeuralODE:

1. Starting from $u_0 = [1, 0]$, predict the trajectory at $t = 0, 0.5, 1.0, 1.5, 2.0$ using your model. Store as `predicted_trajectory` (list of 5 tensors, each shape `(2,)`).

2. Compute the true trajectory at the same time points using `euler_integrate` with the true `spiral_dynamics` and $N = 2000$. Store as `true_trajectory`.

3. Compute the trajectory error at each time point: $e(t) = \|\hat{u}(t) - u(t)\|_2$. Store as `errors` (list of 5 floats).

4. Store the maximum error across all time points as `max_error`.

In [ ]:
### WRITE YOUR SOLUTION HERE ###

u0_test = torch.tensor([[1.0, 0.0]])
time_points = [0.0, 0.5, 1.0, 1.5, 2.0]

predicted_trajectory = ...  # list of 5 tensors, each (2,)
true_trajectory = ...       # list of 5 tensors, each (2,)
errors = ...                # list of 5 floats
max_error = ...             # float

In [ ]:
""" END OF THIS PART """
assert len(predicted_trajectory) == 5
assert len(true_trajectory) == 5
assert len(errors) == 5

# At t=0, both should equal u0
assert torch.allclose(predicted_trajectory[0], torch.tensor([1.0, 0.0]), atol=0.01)
assert torch.allclose(true_trajectory[0], torch.tensor([1.0, 0.0]), atol=0.01)
assert errors[0] < 0.01, "Error at t=0 should be ~0"

print(f"Part 6 passed! Max trajectory error: {max_error:.4f}")
for i, t in enumerate(time_points):
    print(f"  t={t:.1f}: pred={predicted_trajectory[i].detach().tolist()}, "
          f"true={true_trajectory[i].detach().tolist()}, error={errors[i]:.4f}")